# 🚀 Gollek AI Platform - Pure Java Jupyter Notebook

**NO PYTHON** - 100% Java for AI inference using the Gollek SDK!

## Prerequisites

The Gollek server must be running before executing cells:

```bash
cd gollek/ui/gollek-server-runtime
java -jar target/quarkus-app/quarkus-run.jar
# Server starts on http://localhost:8080
```

## Why remote mode?
The `GollekClient.builder().local()` path requires a running Quarkus/CDI container
(`io.quarkus.arc.Arc`) in the same JVM — not available in a plain JShell/IJava kernel.
Remote mode connects to the Gollek server over HTTP, which works perfectly in notebooks.

In [1]:
// Cell 1: Verify Java version
System.out.println("Java: " + Runtime.version());
System.out.println("✅ IJava kernel ready");

Java: 25.0.2+10-LTS-jvmci-b01
✅ IJava kernel ready


In [2]:
// Cell 2: Imports
import tech.kayys.gollek.sdk.GollekClient;
import tech.kayys.gollek.sdk.model.GenerationRequest;
import tech.kayys.gollek.sdk.model.GenerationResponse;
import tech.kayys.gollek.sdk.model.GenerationStream;
import tech.kayys.gollek.sdk.model.EmbeddingRequest;
import tech.kayys.gollek.sdk.model.EmbeddingResponse;
import tech.kayys.gollek.sdk.model.ModelInfo;
import java.util.List;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.TimeUnit;

System.out.println("✅ Imports OK");

✅ Imports OK


In [3]:
// Cell 3: Connect to Gollek server (remote mode)
// Change GOLLEK_URL if your server runs on a different host/port
String GOLLEK_URL = "http://localhost:8080";

GollekClient client = GollekClient.builder()
    .endpoint(GOLLEK_URL)
    .build();

System.out.println("✅ Connected to Gollek server at " + GOLLEK_URL);

✅ Connected to Gollek server at http://localhost:8080


In [4]:
// Cell 4: List available models
List<ModelInfo> models = client.listModels();
if (models.isEmpty()) {
    System.out.println("⚠️  No models found. Pull a model via the Gollek CLI first.");
} else {
    System.out.println("📦 Available models:");
    models.forEach(m -> System.out.println("  - " + m.id() + " (" + m.format() + ")"));
}

EvalException: Model list request failed

In [ ]:
// Cell 5: Simple text generation (blocking)
// Change MODEL to one listed above
String MODEL = "qwen2.5-0.5b-instruct-q4_0";

GenerationResponse response = client.generate(
    GenerationRequest.builder()
        .model(MODEL)
        .prompt("What is the capital of Indonesia? Answer in one sentence.")
        .maxTokens(100)
        .temperature(0.7f)
        .build()
);

System.out.println("🤖 " + response.text());

In [ ]:
// Cell 6: Streaming generation
String MODEL = "qwen2.5-0.5b-instruct-q4_0";
CountDownLatch latch = new CountDownLatch(1);

System.out.print("🌊 ");
try (GenerationStream stream = client.generateStream(
        GenerationRequest.builder()
            .model(MODEL)
            .prompt("Write a haiku about Java programming.")
            .maxTokens(80)
            .stream(true)
            .build())) {
    stream
        .onToken(t -> { System.out.print(t); System.out.flush(); })
        .onComplete(latch::countDown)
        .onError(e -> { System.err.println("\n❌ " + e.getMessage()); latch.countDown(); });
    latch.await(60, TimeUnit.SECONDS);
}
System.out.println("\n✅ Done");

In [ ]:
// Cell 7: Async generation
import java.util.concurrent.CompletableFuture;
String MODEL = "qwen2.5-0.5b-instruct-q4_0";

CompletableFuture<GenerationResponse> future = client.generateAsync(
    GenerationRequest.builder()
        .model(MODEL)
        .prompt("List 3 benefits of Java for AI development.")
        .maxTokens(150)
        .build()
);

System.out.println("⏳ Waiting...");
System.out.println(future.get(60, TimeUnit.SECONDS).text());

In [ ]:
// Cell 8: Embeddings
String MODEL = "qwen2.5-0.5b-instruct-q4_0";

EmbeddingResponse emb = client.embed(
    EmbeddingRequest.builder()
        .model(MODEL)
        .input("Gollek is a high-performance Java AI inference engine")
        .build()
);

float[] v = emb.vector();
System.out.println("🔢 Dimensions: " + v.length);
System.out.printf("First 5: [%.4f, %.4f, %.4f, %.4f, %.4f]%n", v[0], v[1], v[2], v[3], v[4]);

In [ ]:
// Cell 9: Benchmark
String MODEL = "qwen2.5-0.5b-instruct-q4_0";
String[] prompts = {"Say hello.", "What is 2+2?", "Name a color."};

System.out.printf("%-30s | %s%n", "Prompt", "ms");
System.out.println("-".repeat(42));
for (String p : prompts) {
    long t = System.currentTimeMillis();
    client.generate(GenerationRequest.builder().model(MODEL).prompt(p).maxTokens(20).build());
    System.out.printf("%-30s | %d%n", p, System.currentTimeMillis() - t);
}

In [ ]:
// Cell 10: Cleanup
client.close();
System.out.println("✅ Done — Pure Java AI, no Python required! 🚀");